# 03. Обучение по единому протоколу
Тот же модуль, что CLI tsprep-study. Test-сегменты не пересекаются; preprocessing fit выполняется на train. β выбирается по validation на первом seed и фиксируется для повторов. Полный запуск: 144 выбранные модели и 24 дополнительных кандидата β; до 30 эпох.

Новый формат v3: неизменяемые входы, проверяемые receipts, строгий deterministic mode. Старые v2 study не продолжаются; для нового запуска нужен новый каталог.


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "tsprep").is_dir() and (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Run this notebook inside the tsprep project")
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(2)


In [ ]:
from datetime import datetime, timezone
from tsprep.study import run_study
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 30
SEEDS = [42, 43, 44]
DATASETS = ["synthetic", "aapl", "btc"]
STUDY_DIR = ROOT / "results" / datetime.now(timezone.utc).strftime("study_%Y%m%dT%H%M%S%fZ")
print(STUDY_DIR, DEVICE)


In [ ]:
# Each experiment stores full config, fitted preprocessing, inputs, hashes and weights.
# To resume, set STUDY_DIR to an existing study with exactly the same protocol/source.
run_study(ROOT, STUDY_DIR, epochs=EPOCHS, seeds=SEEDS, datasets=DATASETS, device=DEVICE, data_seed=2026, deterministic=True)


In [ ]:
import pandas as pd
runs = pd.read_csv(STUDY_DIR / "comparison_runs.csv")
runs.groupby(["dataset", "model"])[["mse", "forecast_mae", "forecast_vs_mean", "forecast_vs_median", "forecast_vs_raw", "forecast_vs_pca", "mse_vs_pca"]].agg(["mean", "std"])


Среднее/SD по seed и folds — описательная сводка, не доверительный интервал по независимым окнам. Положительное forecast_vs_mean / mse_vs_pca означает проигрыш baseline. Старые артефакты не смешиваются с новым исследованием.
